# Notebook starter

Runs `hello_world` and sets up the imports that every notebook in this repo
needs. It exists to be borrowed from: copy the path cell into a new notebook
and the rest of `training` imports normally.

## Running it

In VS Code, pick the `.venv` interpreter and run the cells.
[.vscode/settings.json](../.vscode/settings.json) sets
`jupyter.notebookFileRoot` to the workspace folder, so the kernel starts at the
repo root.

From the terminal, with `UV_ENV_FILE=".env"` exported as in the README:

```bash
# Execute every cell and print the outputs, leaving the file untouched
uv run jupyter nbconvert --execute --to markdown --stdout \
    notebooks/hello_world.ipynb

# Execute every cell and save the outputs back into the notebook
uv run jupyter nbconvert --execute --to notebook --inplace \
    notebooks/hello_world.ipynb
```

## Put `app/src` on the path

[.env](../.env) sets `PYTHONPATH=app/src`, which is a **relative** path: Python
resolves it against the kernel's working directory, and that directory depends on
who launched the kernel. VS Code starts it at the repo root, where `app/src`
resolves. `jupyter nbconvert --execute` starts it in this notebook's folder,
where it resolves to the non-existent `notebooks/app/src` and every import below
fails.

So the cell below walks up from the working directory until it finds `app/src`
and puts the absolute path on `sys.path`. That works under either launcher, and
it does not care whether `UV_ENV_FILE` was exported.

In [1]:
import sys
from pathlib import Path

repo_root = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "app" / "src").is_dir()
)
sys.path.insert(0, str(repo_root / "app" / "src"))  # noqa: NAR001

print(f"imports resolve from {repo_root / 'app' / 'src'}")  # noqa: NAR001

imports resolve from /Users/anantsimran/code/dawnshard/app/src


## Run `hello_world`

[app/src/hello_world.py](../app/src/hello_world.py) is a single top-level
`print`, so importing it is what runs it. Python caches the module, so re-running
this cell prints nothing the second time — restart the kernel, or call
`importlib.reload(hello_world)`, to see it again.

In [2]:
import hello_world  # noqa: F401

Hello world


## Import from the `training` package

The real check: packages under `app/src/training` import by the same names the
scripts and tests use, so code moves between a notebook and the repo unchanged.

In [3]:
from training.common import constants

print(f"protected token ids: {constants.PROTECTED_IDS}")  # noqa: NAR001

protected token ids: [256, 257, 258, 259]


## Pick a device

`training.setup` chooses CUDA → MPS → CPU at import time and exposes the result
as `DEVICE`. Set `DISABLE_GPU=1` before the import to force CPU. This cell also
imports torch, so it is the slow one.

In [4]:
from training.setup import DEVICE

print(f"device: {DEVICE}")  # noqa: NAR001

device: mps
